In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp1_storage")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cpu_model': 'Apple M4', 'ram': '16.0 GiB', 'cbc': 'Version: 2.10.3', 'neo4j_server': 'Neo4j Kernel 5.26.28 (community)', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
df = pd.DataFrame(result["data"]["rows"])
display(df.pivot_table(index=["n_people", "hops"], columns="backend",
                       values="median_ms").round(3))

backend        memory   neo4j  sqlite
n_people hops                        
50       1      0.039   1.504   0.043
         2      0.054   1.183   0.105
         3      0.074   2.182   0.281
         4      0.093   3.787   0.597
100      1      0.036   0.808   0.058
         2      0.059   1.421   0.141
         3      0.084   2.863   0.398
         4      0.110   4.737   0.971
200      1      0.040   0.689   0.088
         2      0.062   1.849   0.177
         3      0.095   3.756   0.535
         4      0.142   8.718   1.544
300      1      0.096   0.695   0.218
         2      0.071   1.708   0.255
         3      0.114   5.206   0.774
         4      0.174  12.091   2.254
500      1      0.048   0.570   0.172
         2      0.077   1.461   0.269
         3      0.118   3.895   0.633
         4      0.191  10.962   1.963
1000     1      0.064   0.644   0.350
         2      0.103   1.688   0.468
         3      0.162   5.537   0.986
         4      0.268  16.992   2.978

In [3]:
plots.exp1_crossover(result)   # figures/exp1_crossover.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp1_crossover.png')

In [4]:
from experiments import report as _report
from IPython.display import Markdown, display

d = result["data"]
rows = d["rows"]
floor = d.get("protocol_floor")

headline = _report._exp1_headline(rows)
ipc = _report._ipc_dominance(rows, floor["median_ms"]) if floor else None
ratio4 = _report._neo4j_sqlite_ratio_at_hops4(rows)
bad_parity = [p for p in d["parity"] if not p["match"]]
skipped = d["skipped"]
dip_note = _report._result_count_monotonicity_note(rows)

# 최종 리뷰 Important 11: 이 셀은 원래 markdown cell로, 31.4%/0.9169 같은 수치를
# 손으로 타이핑해 두고 있었다(Task 9가 report.py에서 이미 고친 문제가 노트북에는
# 남아 있었음). 이제 위 코드 셀에서 로드한 result["data"]에서 직접 계산해 표시한다
# -- experiments/report.py의 헬퍼를 그대로 재사용해 리포트와 노트북이 같은 소스에서
# 같은 로직으로 같은 문장을 만든다.
lines = ["## 관찰 요약", ""]
if headline:
    lines.append(f"- {headline}")
if floor and ipc:
    lo, hi = ipc
    pct = f"{lo:.0f}%" if lo == hi else f"{lo:.0f}~{hi:.0f}%"
    lines.append(f"- hops=1 지연의 {pct}는 순수 IPC/프로토콜 바닥값(`protocol_floor.median_ms`"
                 f"≈{floor['median_ms']:.3f}ms)이 차지한다 — 얕은 hop 비교는 그래프 순회 "
                 "비용보다는 bolt 왕복 비용을 주로 재는 셈이라, hops=1만 보고 백엔드를 비교하는 "
                 "것은 오해를 부른다.")
if ratio4:
    lo, hi = ratio4
    ratio_str = f"{lo:.2f}배" if lo == hi else f"{lo:.2f}~{hi:.2f}배"
    lines.append(f"- neo4j/sqlite 배율은 hops=4 기준 {ratio_str} 범위에서 평평하며, 규모가 "
                 "커진다고 좁혀지는 추세는 관측되지 않았다.")
lines.append(f"- `data.parity`는 "
             f"{'전 조합에서 match: true' if not bad_parity else f'{len(bad_parity)}건 불일치'}"
             "이므로 위 지연 차이는 결과 집합 크기 차이가 아니라 백엔드 자체의 지연 특성이다. "
             f"`data.skipped`는 "
             f"{'비어 있다(neo4j 측정 전 구간 성공)' if not skipped else f'{len(skipped)}건 기록됨'}.")
if dip_note:
    lines.append(dip_note)

display(Markdown("\n".join(lines)))

## 관찰 요약

- **측정 범위(50~1000명, hops 1~4) 전 구간에서 memory < sqlite < neo4j 순서가 유지된다 — "규모·hops가 커지면 Neo4j가 역전한다"는 이 실험의 원 가설은 기각된다.**
- hops=1 지연의 40~106%는 순수 IPC/프로토콜 바닥값(`protocol_floor.median_ms`≈0.606ms)이 차지한다 — 얕은 hop 비교는 그래프 순회 비용보다는 bolt 왕복 비용을 주로 재는 셈이라, hops=1만 보고 백엔드를 비교하는 것은 오해를 부른다.
- neo4j/sqlite 배율은 hops=4 기준 4.88~6.34배 범위에서 평평하며, 규모가 커진다고 좁혀지는 추세는 관측되지 않았다.
- `data.parity`는 전 조합에서 match: true이므로 위 지연 차이는 결과 집합 크기 차이가 아니라 백엔드 자체의 지연 특성이다. `data.skipped`는 비어 있다(neo4j 측정 전 구간 성공).
> 인원 수가 늘어난다고 순회 결과 수(result_count)가 항상 함께 늘지는 않는다 — hops=1: n=100→200에서 결과수 19→17; hops=1: n=300→500에서 결과수 26→16; hops=2: n=300→500에서 결과수 130→94; hops=3: n=300→500에서 결과수 459→330; hops=4: n=300→500에서 결과수 1038→990. 인원 수를 작업량의 단조 대리지표로 쓰지 말 것(결과 수가 적으면 순회할 것도 적어 그만큼 더 빠르다).